In [4]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load pre-trained model (weights)
with torch.no_grad():
        model = GPT2LMHeadModel.from_pretrained('gpt2')
        model.eval()
# Load pre-trained model tokenizer (vocabulary)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

# Predicting the next token

In [5]:
def predict_top_k_next_tokens(model, tokenizer, context_string, k=10):
    # Encode the context string
    input_ids = tokenizer.encode(context_string, return_tensors='pt')

    # Get model predictions
    with torch.no_grad():
        outputs = model(input_ids)
        predictions = outputs.logits

    # Get the predictions for the last token
    last_token_logits = predictions[0, -1, :]

    # Get the top k predicted tokens and their probabilities
    top_k_probs, top_k_indices = torch.topk(last_token_logits, k=k)

    # Decode the top k token indices to human-readable tokens
    predicted_tokens = [tokenizer.decode([idx]) for idx in top_k_indices]

    # Return a list of (token, probability) tuples
    return list(zip(predicted_tokens, torch.nn.functional.softmax(top_k_probs, dim=0).numpy()))


In [6]:
# Example usage:
context = "The governor ordered more sandbags to prepare for the"
top_10_tokens = predict_top_k_next_tokens(model, tokenizer, context)

print(f"Top 10 predicted next tokens for '{context}':")
for token, prob in top_10_tokens:
    print(f"- '{token}' (Probability: {prob:.4f})")


Top 10 predicted next tokens for 'The governor ordered more sandbags to prepare for the':
- ' storm' (Probability: 0.6850)
- ' upcoming' (Probability: 0.0526)
- ' coming' (Probability: 0.0456)
- ' arrival' (Probability: 0.0369)
- ' flood' (Probability: 0.0356)
- ' storms' (Probability: 0.0352)
- ' worst' (Probability: 0.0339)
- ' next' (Probability: 0.0282)
- ' flooding' (Probability: 0.0238)
- ' event' (Probability: 0.0233)


# Scoring the probabilities of sentences

In [7]:
def score(sentence, tokenizer, model):
    tokenize_input = tokenizer.encode(sentence)
    tensor_input = torch.tensor([tokenize_input])
    loss = model(tensor_input, labels=tensor_input)[0]
    return -loss.detach().numpy()

In [16]:
score('May your woes be many, and your days few', tokenizer, model)

np.float32(-4.93779)

In [17]:
score('May you\'re woes be many, and your\'re day fewest', tokenizer, model)

np.float32(-7.583763)

In [10]:
score('The president ceased trading with China yesterday', tokenizer, model)

np.float32(-6.8108163)

In [11]:
score('The president ceases trading to China yesterday', tokenizer, model)

np.float32(-7.9418874)

In [12]:
score("My favorite things are eating my family and not using punctuation", tokenizer, model)

np.float32(-5.1933236)

In [13]:
score("Colorless green ideas sleep furiously", tokenizer, model)

np.float32(-10.089575)